# Historical experiment: model_to_c.ipynb
Archived source with outputs/execution counts cleared. Superseded by the numbered notebooks.
Original paths and experimental branches are retained for reference; this is not the supported run sequence.

In [ ]:
from pathlib import Path
import numpy as np

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite import Interpreter

In [ ]:
model_path = Path(
    "/home/randy/Documents/ACompArc/model/project/block0_w8a8.tflite"
)

In [ ]:
interpreter = Interpreter(
    model_path=str(model_path),
    experimental_preserve_all_tensors=True,
)
interpreter.allocate_tensors()

In [ ]:
tensors = {
    t["index"]: t
    for t in interpreter.get_tensor_details()
}

In [ ]:
def describe_tensor(index):
    if index < 0:
        print("    Optional tensor: absent")
        return

    t = tensors[index]
    q = t["quantization_parameters"]

    print(
        f"    T{index}: {t['name']}\n"
        f"      shape={t['shape'].tolist()}, "
        f"dtype={np.dtype(t['dtype']).name}"
    )
    print(
        f"      scales={q['scales'].tolist()}\n"
        f"      zero_points={q['zero_points'].tolist()}\n"
        f"      quantized_dimension={q['quantized_dimension']}"
    )

In [ ]:
print("MODEL INPUTS")
for t in interpreter.get_input_details():
    describe_tensor(t["index"])

In [ ]:
# Private inspection API: useful for examining this local export.
print("\nMATRIX OPERATIONS")
for op in interpreter._get_ops_details():
    if op["op_name"] in ("FULLY_CONNECTED", "BATCH_MATMUL"):
        print(f"\nOp {op['index']}: {op['op_name']}")

        print("  Inputs:")
        for index in op["inputs"]:
            describe_tensor(int(index))

        print("  Outputs:")
        for index in op["outputs"]:
            describe_tensor(int(index))

# EXPORT THE WEIGHT TO NPZ

In [ ]:
from pathlib import Path
import numpy as np

try:
    from ai_edge_litert.interpreter import Interpreter, OpResolverType
except ImportError:
    from tensorflow.lite import Interpreter
    from tensorflow.lite.experimental import OpResolverType

project_dir = Path(
    "/home/randy/Documents/ACompArc/model/project"
)

reference = Interpreter(
    model_path=str(project_dir / "block0_w8a8.tflite"),
    experimental_preserve_all_tensors=True,
    experimental_op_resolver_type=OpResolverType.BUILTIN_REF,
)
reference.allocate_tensors()

# Confirm we are extracting the operation identified in your output.
op = next(
    op for op in reference._get_ops_details()
    if op["index"] == 15
)
assert op["op_name"] == "FULLY_CONNECTED"
assert list(op["inputs"]) == [45, 27, -1]
assert list(op["outputs"]) == [46]

inputs = reference.get_input_details()
assert len(inputs) == 1
assert inputs[0]["shape"].tolist() == [1, 32, 128]
assert inputs[0]["dtype"] == np.float32

# Deterministic synthetic block input for the first arithmetic test.
# Later, repeat with real hidden states from your token-based test.
rng = np.random.default_rng(42)
block_input = rng.standard_normal((1, 32, 128)).astype(np.float32)

reference.set_tensor(inputs[0]["index"], block_input)
reference.invoke()

# Capture the projection's input AFTER the preceding operations.
x_q = reference.get_tensor(45).copy()
w_q = reference.get_tensor(27).copy()
y_ref = reference.get_tensor(46).copy()

assert x_q.dtype == w_q.dtype == y_ref.dtype == np.int8
assert x_q.shape == y_ref.shape == (1, 32, 128)
assert w_q.shape == (128, 128)

details = {
    t["index"]: t
    for t in reference.get_tensor_details()
}

qx = details[45]["quantization_parameters"]
qw = details[27]["quantization_parameters"]
qy = details[46]["quantization_parameters"]

zx = int(qx["zero_points"][0])
assert np.all(qw["zero_points"] == 0)

# Mathematical INT32 accumulator reference, before requantization.
# Cast BEFORE subtracting the zero point to avoid INT8 overflow.
x_centered = x_q.reshape(32, 128).astype(np.int32) - zx
acc_ref = x_centered @ w_q.astype(np.int32).T

export_dir = project_dir / "exports"
export_dir.mkdir(exist_ok=True)
export_path = export_dir / "q_proj_case0.npz"

np.savez(
    export_path,
    block_input=block_input,
    x_q=x_q,
    w_q=w_q,
    acc_ref=acc_ref,
    y_ref=y_ref,
    x_scales=qx["scales"],
    x_zero_points=qx["zero_points"],
    w_scales=qw["scales"],
    w_zero_points=qw["zero_points"],
    w_quantized_dimension=np.array(qw["quantized_dimension"]),
    y_scales=qy["scales"],
    y_zero_points=qy["zero_points"],
)

print("Saved:", export_path)
print("x_q:", x_q.shape, x_q.dtype)
print("w_q:", w_q.shape, w_q.dtype)
print("acc_ref:", acc_ref.shape, acc_ref.dtype)
print("y_ref:", y_ref.shape, y_ref.dtype)
print("First 8 accumulators:", acc_ref[0, :8])
print("First 8 reference outputs:", y_ref[0, 0, :8])

# CREATE THE C FILE AND HEADER

In [ ]:
from pathlib import Path
import numpy as np

folder = Path(
    "/home/randy/Documents/ACompArc/model/project/exports"
)

with np.load(folder / "q_proj_case0.npz") as data:
    x = data["x_q"].reshape(32, 128)
    w = data["w_q"]
    expected = data["acc_ref"]
    zx = int(data["x_zero_points"][0])

    assert x.shape == (32, 128)
    assert w.shape == (128, 128)
    assert expected.shape == (32, 128)
    assert np.all(data["w_zero_points"] == 0)

def c_array(name, dtype, values):
    flat = values.reshape(-1)
    lines = [
        "    " + ", ".join(str(int(v)) for v in flat[i:i+16])
        for i in range(0, flat.size, 16)
    ]
    return (
        f"static const {dtype} {name}[{flat.size}] = {{\n"
        + ",\n".join(lines) + "\n};\n"
    )

header = (
    "#ifndef QPROJ_DATA_H\n"
    "#define QPROJ_DATA_H\n"
    "#include <stdint.h>\n"
    f"#define INPUT_ZERO_POINT ({zx})\n\n"
)
header += c_array("input", "int8_t", x)
header += c_array("weights", "int8_t", w)
header += c_array("expected_acc", "int32_t", expected)
header += "#endif\n"

(folder / "qproj_data.h").write_text(header)

source = r'''
#include <stdint.h>
#include <stdio.h>
#include "qproj_data.h"

static int32_t output[32 * 128];

__attribute__((noinline))
void qproj_accumulate(void)
{
    for (int t = 0; t < 32; ++t) {
        for (int o = 0; o < 128; ++o) {
            int32_t acc = 0;

            for (int k = 0; k < 128; ++k) {
                int32_t x =
                    (int32_t)input[t * 128 + k]
                    - INPUT_ZERO_POINT;

                int32_t w =
                    (int32_t)weights[o * 128 + k];

                acc += x * w;
            }

            output[t * 128 + o] = acc;
        }
    }
}

int main(void)
{
    qproj_accumulate();

    int errors = 0;
    for (int i = 0; i < 32 * 128; ++i) {
        if (output[i] != expected_acc[i])
            ++errors;
    }

    printf("Checked 4096 accumulators; mismatches: %d\n", errors);
    printf("%s\n", errors == 0 ? "PASS" : "FAIL");

    return errors == 0 ? 0 : 1;
}
'''

(folder / "qproj_acc_test.c").write_text(source)
print("Created qproj_data.h and qproj_acc_test.c")

In [ ]:
with np.load(folder / "q_proj_case0.npz") as data:
    sx = float(data["x_scales"][0])
    sw = data["w_scales"].astype(np.float64)
    sy = float(data["y_scales"][0])

    M = sx * sw / sy

print("Multiplier count:", M.size)
print("First 4 multipliers:", M[:4])

In [ ]:
with np.load(folder / "q_proj_case0.npz") as data:
    print("Weight shape:", data["w_q"].shape)
    print("Weight scales:", data["w_scales"])

In [ ]:
import numpy as np

# Map tensor indices to their metadata.
tensor_by_index = {
    int(t["index"]): t
    for t in interpreter.get_tensor_details()
}

def calculate_multipliers(op):
    index_x, index_w = map(int, op["inputs"][:2])
    index_y = int(op["outputs"][0])

    def scales(index):
        return np.asarray(
            tensor_by_index[index]["quantization_parameters"]["scales"],
            dtype=np.float64,
        )

    sx = scales(index_x)
    sw = scales(index_w)
    sy = scales(index_y)

    # This helper assumes one scale each for input and output.
    if sx.size != 1 or sy.size != 1 or sw.size == 0:
        raise ValueError("Expected scalar input/output scales and weight scales.")

    M = sx[0] * sw / sy[0]

    print(f"Op {op['index']}: {op['op_name']}")
    print(f"T{index_x} × T{index_w} → T{index_y}")
    print(f"Weight scale count: {sw.size}")

    if np.all(M == M[0]):
        print(f"Shared multiplier: {M[0]:.12g}")
    else:
        print("Multipliers per output feature:", M)

    return M

In [ ]:
op15 = next(
    op for op in interpreter._get_ops_details()
    if int(op["index"]) == 15
)

M = calculate_multipliers(op15)

In [ ]:
from pathlib import Path
import numpy as np

folder = Path("/home/randy/Documents/ACompArc/model/project/exports")

with np.load(folder / "q_proj_case0.npz") as data:
    expected_y = data["y_ref"].reshape(-1)
    assert expected_y.dtype == np.int8
    assert expected_y.size == 4096

values = ", ".join(str(int(v)) for v in expected_y)

(folder / "qproj_y_ref.h").write_text(
    "#pragma once\n"
    "#include <stdint.h>\n"
    f"static const int8_t expected_y[4096] = {{\n{values}\n}};\n"
)